# Gold — Customer Summary MERGE

Lifetime customer metrics with Delta **MERGE**: insert new customers, update changed metrics, soft-delete (mark inactive) when `last_order_date` is before the inactivity cutoff.

This notebook runs **two merges** to demonstrate insert counts (first load) then update/soft-delete counts (stricter cutoff).

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.gold.customer_summary as summary_module
importlib.reload(summary_module)

from src.gold.customer_summary import (
    GoldCustomerSummaryConfig,
    build_customer_summary_source,
    run_customer_summary_merge,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = GoldCustomerSummaryConfig()
print("Loaded from:", summary_module.__file__)

## Pass 1 — initial load (all customers active)

Uses a very large inactivity window so every customer is marked active on first MERGE.

In [ ]:
from pyspark.sql import functions as F

pass1_config = GoldCustomerSummaryConfig(inactive_days=9999)
spark.sql(f"DROP TABLE IF EXISTS {pass1_config.target_table}")
pass1 = run_customer_summary_merge(spark, pass1_config)
print(pass1)
display(spark.table(pass1_config.target_table).orderBy(F.col("total_spend").desc()).limit(10))

## Pass 2 — stricter cutoff (soft-delete inactive customers)

Customers with no delivered order in the last **180 days** before the dataset end date are marked inactive.

In [ ]:
from pyspark.sql import functions as F

pass2_config = GoldCustomerSummaryConfig(inactive_days=180)
pass2 = run_customer_summary_merge(spark, pass2_config)
print(pass2)
display(
    spark.table(pass2_config.target_table)
    .groupBy("is_active")
    .count()
    .orderBy("is_active")
)

In [ ]:
import json

report = {
    "task": "gold_customer_summary_merge",
    "target_table": config.target_table,
    "pass1_initial_load": pass1,
    "pass2_inactivity_refresh": pass2,
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/gold_customer_summary.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)